In [1]:
import os
from datetime import datetime


def read_errors(path, stats=None):
    if not path or type(path) != str:
        raise ValueError("File path valid string honi chahiye")
    if not os.path.exists(path):
        raise FileNotFoundError(f"Log file nahi mili: {path}")

    if stats is not None:
        stats["malformed_count"] = 0

    with open(path, "r", encoding="utf-8") as file:
        line_number = 0
        for raw_line in file:
            line_number += 1
            line = raw_line.strip()

            # Empty lines ko malformed count karein ya skip karein
            if not line:
                if stats is not None:
                    stats["malformed_count"] += 1
                continue

            parts = line.split(" | ", 2)
            if len(parts) != 3:
                if stats is not None:
                    stats["malformed_count"] += 1
                continue

            timestamp, level, message = parts[0].strip(), parts[1].strip(), parts[2].strip()

            # Timestamp format validate karein (YYYY-MM-DD HH:MM:SS)
            try:
                datetime.strptime(timestamp, "%Y-%m-%d %H:%M:%S")
            except ValueError:
                if stats is not None:
                    stats["malformed_count"] += 1
                continue

            # Level aur message khali nahi hone chahiye
            if not level or not message:
                if stats is not None:
                    stats["malformed_count"] += 1
                continue

            if level == "ERROR":
                yield (line_number, timestamp, message)


def summarize_errors(path):
    stats = {"malformed_count": 0}
    counts_by_date = {}

    for line_num, timestamp, message in read_errors(path, stats=stats):
        date_part = timestamp.split(" ")[0]
        if date_part not in counts_by_date:
            counts_by_date[date_part] = 0
        counts_by_date[date_part] += 1

    return {
        "errors_by_date": counts_by_date,
        "malformed_lines": stats["malformed_count"]
    }


# Sample log file generation & testing
if __name__ == "__main__":
    sample_file = "sample_app.log"
    sample_logs = [
        "2026-11-07 10:30:00 | ERROR | Connection lost",
        "2026-11-07 10:31:15 | INFO | User logged in",
        "This is a corrupted log line without pipes",
        "2026-11-07 11:05:22 | ERROR | Database timeout",
        "invalid-date | ERROR | Bad timestamp format",
        "2026-11-08 09:12:45 | WARNING | High memory usage",
        "2026-11-08 14:20:10 | ERROR | Disk write failed"
    ]

    with open(sample_file, "w", encoding="utf-8") as f:
        for entry in sample_logs:
            f.write(entry + "\n")

    print("Streamed Errors:")
    tracker = {}
    for err in read_errors(sample_file, stats=tracker):
        print(err)
    print("Malformed lines skipped:", tracker["malformed_count"])

    print("\nSummary:")
    print(summarize_errors(sample_file))

Streamed Errors:
(1, '2026-11-07 10:30:00', 'Connection lost')
(4, '2026-11-07 11:05:22', 'Database timeout')
(7, '2026-11-08 14:20:10', 'Disk write failed')
Malformed lines skipped: 2

Summary:
{'errors_by_date': {'2026-11-07': 2, '2026-11-08': 1}, 'malformed_lines': 2}
